# 03 - Change-Point Detection

Identifies structural breaks in the monthly inundation extent and NDVI time series using the PELT algorithm.

**Method:** Pruned Exact Linear Time (PELT) with Gaussian (L2) cost, penalty = log(n) Ã— ÏƒÂ², minimum segment = 6 months.  
**Applied to:** Raw monthly series AND deseasonalised anomaly series.  
**Output:** Breakpoint date saved to `data/processed/breakpoint.json` for use by downstream notebooks.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
import json
import ruptures as rpt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Publication style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'figure.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PROCESSED_DIR = Path('data/processed')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

ANALYSIS_START = '2019-01'
ANALYSIS_END = '2024-12'

## Load Data and Apply Wetland Mask

In [ ]:
# Load cubes
inundation_cube = xr.open_dataarray(PROCESSED_DIR / 'inundation_cube.nc')
ndvi_cube = xr.open_dataarray(PROCESSED_DIR / 'ndvi_cube.nc')

# Filter to analysis period 2019-2024
inundation_cube = inundation_cube.sel(time=slice(ANALYSIS_START, ANALYSIS_END))
ndvi_cube = ndvi_cube.sel(time=slice(ANALYSIS_START, ANALYSIS_END))

# Load wetland mask
with rasterio.open('data/raw/wetland_mask.tif') as src:
    mask_arr = src.read(1)  # 1 = wetland

# Align mask to cube spatial dims (same grid from preprocessing)
wetland_mask = xr.DataArray(mask_arr == 1, dims=['y', 'x'],
                             coords={'y': inundation_cube.y, 'x': inundation_cube.x})

print(f"Inundation cube: {inundation_cube.shape}, time: {inundation_cube.time.values[0]} to {inundation_cube.time.values[-1]}")
print(f"NDVI cube: {ndvi_cube.shape}")
print(f"Wetland pixels: {int(wetland_mask.sum()):,}")

## Compute Monthly Aggregate Time Series

In [ ]:
# Mask to wetland only
inund_masked = inundation_cube.where(wetland_mask)

# Valid pixels (not nodata) per month
valid = (inund_masked >= 0)
water = (inund_masked == 1)

# Monthly inundation extent: % of valid wetland pixels that are water
monthly_inundation = (water.sum(dim=['x', 'y']) / valid.sum(dim=['x', 'y']) * 100).to_series()
monthly_inundation.index = pd.to_datetime(monthly_inundation.index)

# Monthly mean NDVI within wetland mask (NaN for missing months)
ndvi_masked = ndvi_cube.where(wetland_mask & (ndvi_cube > -1))

# NDVI cube may have fewer time steps â€” build full monthly index
full_time = pd.date_range('2019-01', '2024-12', freq='MS')
ndvi_series = pd.Series(np.nan, index=full_time)
ndvi_vals = ndvi_masked.mean(dim=['x', 'y']).to_series()
ndvi_vals.index = pd.to_datetime(ndvi_vals.index)
ndvi_series.update(ndvi_vals)

print(f"Inundation series: {len(monthly_inundation)} months, range {monthly_inundation.min():.1f}\u2013{monthly_inundation.max():.1f}%")
print(f"NDVI series: {ndvi_series.notna().sum()} valid months out of {len(ndvi_series)}")

## Compute Deseasonalised Anomaly Series

In [ ]:
def deseasonalise(series):
    """Subtract long-term monthly climatological mean."""
    monthly_mean = series.groupby(series.index.month).mean()
    anomaly = series.copy()
    for m in range(1, 13):
        idx = series.index.month == m
        anomaly[idx] = series[idx] - monthly_mean[m]
    return anomaly

inundation_anomaly = deseasonalise(monthly_inundation)

# For NDVI, interpolate missing before deseasonalising, then restore NaNs
ndvi_interp = ndvi_series.interpolate(method='linear', limit_direction='both')
ndvi_anomaly_full = deseasonalise(ndvi_interp)
ndvi_anomaly = ndvi_anomaly_full.copy()
ndvi_anomaly[ndvi_series.isna()] = np.nan

print("Anomaly series computed.")
print(f"Inundation anomaly: mean={inundation_anomaly.mean():.2f}, std={inundation_anomaly.std():.2f}")

## PELT Change-Point Detection

In [ ]:
def run_pelt(series, min_size=6, label=''):
    """
    Run PELT with L2 cost and BIC-like penalty.
    pen = log(n) * sigma^2
    Returns list of breakpoint indices (sample index, 0-based) and corresponding dates.
    """
    # Use only valid (non-NaN) values but keep track of positions
    valid_idx = series.dropna().index
    values = series.dropna().values.astype(float)
    n = len(values)
    sigma2 = np.var(values, ddof=1)
    pen = np.log(n) * sigma2

    algo = rpt.Pelt(model='l2', min_size=min_size).fit(values)
    bkps = algo.predict(pen=pen)
    # bkps[-1] is always n (end of series), remove it
    bkps_interior = bkps[:-1]

    # Map back to dates
    dates = [valid_idx[i - 1] for i in bkps_interior]  # i is 1-based end of segment

    print(f"{label}: n={n}, sigma2={sigma2:.4f}, pen={pen:.4f}")
    print(f"  Breakpoints at indices: {bkps_interior}")
    print(f"  Breakpoint dates: {[str(d.date()) for d in dates]}")
    return bkps_interior, dates, values, valid_idx

# Raw series
inund_bkps_idx, inund_bkps_dates, inund_vals, inund_idx = run_pelt(
    monthly_inundation, min_size=6, label='Inundation (raw)')

ndvi_bkps_idx, ndvi_bkps_dates, ndvi_vals, ndvi_idx = run_pelt(
    ndvi_series, min_size=6, label='NDVI (raw)')

# Anomaly series
inund_anom_bkps_idx, inund_anom_bkps_dates, inund_anom_vals, inund_anom_idx = run_pelt(
    inundation_anomaly, min_size=6, label='Inundation anomaly')

ndvi_anom_bkps_idx, ndvi_anom_bkps_dates, ndvi_anom_vals, ndvi_anom_idx = run_pelt(
    ndvi_anomaly, min_size=6, label='NDVI anomaly')

In [ ]:
# Primary breakpoint: first breakpoint from inundation raw series
primary_breakpoint = inund_bkps_dates[0] if inund_bkps_dates else pd.Timestamp('2022-01-01')
print(f"Primary breakpoint (inundation): {primary_breakpoint.strftime('%Y-%m')}")
print(f"Primary breakpoint (NDVI):       {ndvi_bkps_dates[0].strftime('%Y-%m') if ndvi_bkps_dates else 'None'}")
print(f"Primary breakpoint (inund anom): {inund_anom_bkps_dates[0].strftime('%Y-%m') if inund_anom_bkps_dates else 'None'}")
print(f"Primary breakpoint (NDVI anom):  {ndvi_anom_bkps_dates[0].strftime('%Y-%m') if ndvi_anom_bkps_dates else 'None'}")

# Save for downstream notebooks
bp_data = {
    'primary_breakpoint': primary_breakpoint.strftime('%Y-%m-%d'),
    'inundation_raw_breakpoints': [d.strftime('%Y-%m-%d') for d in inund_bkps_dates],
    'ndvi_raw_breakpoints': [d.strftime('%Y-%m-%d') for d in ndvi_bkps_dates],
    'inundation_anomaly_breakpoints': [d.strftime('%Y-%m-%d') for d in inund_anom_bkps_dates],
    'ndvi_anomaly_breakpoints': [d.strftime('%Y-%m-%d') for d in ndvi_anom_bkps_dates],
}
with open(PROCESSED_DIR / 'breakpoint.json', 'w') as f:
    json.dump(bp_data, f, indent=2)
print(f"\nSaved: {PROCESSED_DIR / 'breakpoint.json'}")

## Statistical Tests — Pre vs Post Period

Mann-Whitney U tests (non-parametric, two-sided) comparing pre- and post-intervention
distributions for both the **raw** series and the **deseasonalised anomaly** series.
The anomaly tests isolate structural shifts in mean level from seasonally-driven variation.

Significance thresholds: \*\*\* p<0.001 · \*\* p<0.01 · \* p<0.05 · n.s. not significant

In [ ]:
from scipy.stats import mannwhitneyu

bp = primary_breakpoint

# Raw series
pre_inund      = monthly_inundation[monthly_inundation.index < bp]
post_inund     = monthly_inundation[monthly_inundation.index >= bp]
pre_ndvi_raw   = ndvi_series[ndvi_series.index < bp].dropna()
post_ndvi_raw  = ndvi_series[ndvi_series.index >= bp].dropna()

# Anomaly series
pre_inund_anom  = inundation_anomaly[inundation_anomaly.index < bp]
post_inund_anom = inundation_anomaly[inundation_anomaly.index >= bp]
pre_ndvi_anom   = ndvi_anomaly[ndvi_anomaly.index < bp].dropna()
post_ndvi_anom  = ndvi_anomaly[ndvi_anomaly.index >= bp].dropna()

tests = [
    ("Inundation extent (raw)",       pre_inund,      post_inund,      "%"),
    ("NDVI (raw)",                    pre_ndvi_raw,   post_ndvi_raw,   ""),
    ("Inundation anomaly (deseas.)",  pre_inund_anom, post_inund_anom, "pp"),
    ("NDVI anomaly (deseas.)",        pre_ndvi_anom,  post_ndvi_anom,  ""),
]

mw_results = {}
print(f"Mann-Whitney U Tests  (breakpoint: {bp.strftime('%Y-%m')})")
print("-" * 85)
print(f"  {'Variable':<38} {'Pre n':>5} {'Post n':>6} {'Pre med':>9} {'Post med':>9} {'U':>9} {'p-value':>12}  Sig")
print("-" * 85)

for label, pre, post, unit in tests:
    u, p = mannwhitneyu(pre.values, post.values, alternative="two-sided")
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
    pm = np.median(pre.values)
    qm = np.median(post.values)
    print(f"  {label:<38} {len(pre):>5} {len(post):>6} {pm:>8.3f}{unit} {qm:>8.3f}{unit} {u:>9.0f} {p:>12.4e}  {sig}")
    mw_results[label] = {"U": u, "p": p, "sig": sig}

print("-" * 85)
print("Significance: *** p<0.001  ** p<0.01  * p<0.05  n.s. not significant")

# Store p-values for figure annotation
p_inund_raw  = mw_results["Inundation extent (raw)"]["p"]
p_ndvi_raw   = mw_results["NDVI (raw)"]["p"]
p_inund_anom = mw_results["Inundation anomaly (deseas.)"]["p"]
p_ndvi_anom  = mw_results["NDVI anomaly (deseas.)"]["p"]

## Figure 1 â€” Time Series with Structural Breakpoints

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

colors = {'pre': '#2166ac', 'post': '#d6604d', 'bp': '#1a1a1a', 'anom': '#4dac26'}

def plot_series_with_bkps(ax, series, bkps_dates, title, ylabel, color, fill_alpha=0.12):
    """Plot a time series with shaded pre/post regions and breakpoint vline."""
    ax.plot(series.index, series.values, color=color, linewidth=1.2, zorder=3)
    
    if bkps_dates:
        bp = bkps_dates[0]
        ax.axvline(bp, color=colors['bp'], linewidth=1.5, linestyle='--', zorder=4, label=f'Breakpoint: {bp.strftime("%Y-%m")}')
        ax.axvspan(series.index[0], bp, alpha=fill_alpha, color=colors['pre'], label='Pre-intervention')
        ax.axvspan(bp, series.index[-1], alpha=fill_alpha, color=colors['post'], label='Post-intervention')
        
        # Mark additional breakpoints if any
        for bp2 in bkps_dates[1:]:
            ax.axvline(bp2, color='grey', linewidth=1, linestyle=':', zorder=4)
    
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=10, pad=4)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    ax.legend(fontsize=8, loc='upper left')

plot_series_with_bkps(axes[0], monthly_inundation, inund_bkps_dates,
                      'Inundation Extent (raw)', 'Extent (%)', '#2166ac')

plot_series_with_bkps(axes[1], ndvi_series, ndvi_bkps_dates,
                      'Mean NDVI within Wetland (raw)', 'NDVI', '#1b7837')

plot_series_with_bkps(axes[2], inundation_anomaly, inund_anom_bkps_dates,
                      'Inundation Extent Anomaly (deseasonalised)', 'Anomaly (pp)', '#4393c3')
axes[2].axhline(0, color='grey', linewidth=0.6, linestyle='-')

plot_series_with_bkps(axes[3], ndvi_anomaly, ndvi_anom_bkps_dates,
                      'NDVI Anomaly (deseasonalised)', 'Anomaly', '#74c476')
axes[3].axhline(0, color='grey', linewidth=0.6, linestyle='-')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].set_xlabel('Date')

plt.suptitle('PELT Change-Point Detection \u2014 Wiang Nong Lom Wetland (2019\u20132024)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_02_changepoint_timeseries.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'fig_02_changepoint_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: outputs/fig_02_changepoint_timeseries.pdf/.png")

## Figure — Pre/Post Distributions with Significance Tests

In [ ]:
def sig_label(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "n.s."

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

datasets = [
    (axes[0, 0], pre_inund,      post_inund,
     "Inundation Extent (%)",      "Monthly Inundation Extent (raw)",         p_inund_raw),
    (axes[0, 1], pre_ndvi_raw,   post_ndvi_raw,
     "NDVI",                        "Mean NDVI within Wetland (raw)",          p_ndvi_raw),
    (axes[1, 0], pre_inund_anom, post_inund_anom,
     "Anomaly (percentage points)", "Inundation Anomaly (deseasonalised)",     p_inund_anom),
    (axes[1, 1], pre_ndvi_anom,  post_ndvi_anom,
     "NDVI Anomaly",                "NDVI Anomaly (deseasonalised)",           p_ndvi_anom),
]

for ax, pre, post, ylabel, title, p_val in datasets:
    bplot = ax.boxplot(
        [pre.values, post.values], patch_artist=True, widths=0.5,
        medianprops=dict(color="white", linewidth=2),
        whiskerprops=dict(linewidth=1), capprops=dict(linewidth=1),
        flierprops=dict(marker="o", markersize=3, alpha=0.4))
    for patch, col in zip(bplot["boxes"], ["#2166ac", "#d6604d"]):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)

    for i, (vals, col) in enumerate([(pre.values, "#2166ac"), (post.values, "#d6604d")], 1):
        jitter = np.random.uniform(-0.1, 0.1, len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, color=col, alpha=0.4, s=12, zorder=5)

    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"Pre\n(n={len(pre)})", f"Post\n(n={len(post)})"]) 
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.25, axis="y")

    # Mean labels
    all_vals = np.concatenate([pre.values, post.values])
    data_range = np.nanmax(all_vals) - np.nanmin(all_vals)
    for i, vals in enumerate([pre.values, post.values], 1):
        ax.text(i, np.nanmax(vals) + data_range * 0.03,
                f"\u03bc={np.nanmean(vals):.3f}", ha="center", fontsize=8, color="#333")

    # Significance bracket
    y_bracket = np.nanmax(all_vals) + data_range * 0.18
    ax.plot([1, 1, 2, 2],
            [y_bracket - data_range*0.02, y_bracket, y_bracket, y_bracket - data_range*0.02],
            color="#333", linewidth=1)
    bracket_label = f"{sig_label(p_val)}  (p = {p_val:.3e})"
    ax.text(1.5, y_bracket + data_range * 0.01, bracket_label,
            ha="center", va="bottom", fontsize=9,
            color="#d62728" if p_val < 0.05 else "grey")
    ax.set_ylim(top=np.nanmax(all_vals) + data_range * 0.38)

plt.suptitle(
    f"Pre vs Post Breakpoint ({bp.strftime('%Y-%m')}) — Mann-Whitney U Tests\n"
    "Raw and Deseasonalised Series",
    fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_03_prepost_boxplot.pdf", bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "fig_03_prepost_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_03_prepost_boxplot.pdf/.png")

## Summary

| Series | Breakpoint(s) |
|--------|--------------|
| Inundation (raw) | See `breakpoint.json` |
| NDVI (raw) | See `breakpoint.json` |
| Inundation (anomaly) | See `breakpoint.json` |
| NDVI (anomaly) | See `breakpoint.json` |

The primary structural break (inundation raw series) is used to define pre/post periods in all downstream analyses.

**Next:** `04_inundation_transition.ipynb`